# Chapter 57: Edge AI & Embedded Computer Vision
## Multimedia & Computer Vision Learning Hub — ITERA
**Author:** Aji Pamungkas Tri Nurcahyo  
**Objective:** Menguasai matematika kuantisasi pasca-pelatihan (*Post-Training Quantization / PTQ*) untuk kompresi model deep learning, menghitung parameter skala $S$ dan titik nol $Z$, mengevaluasi galat dekuantisasi dan SQNR, serta menganalisis efisiensi memori untuk inferensi computer vision pada perangkat keras komputasi tepi (*Edge AI*).

### 1. Landasan Matematis: Kuantisasi Affine Seragam 8-Bit
Persamaan Pemetaan Linier:
$$r \approx S \times (q - Z)$$

Skala Kuantisasi $S$:
$$S = \frac{r_{\max} - r_{\min}}{q_{\max} - q_{\min}} = \frac{r_{\max} - r_{\min}}{255}$$

Titik Nol (Zero-Point) $Z$:
$$Z = \text{clamp}\left(\text{round}\left(-\frac{r_{\min}}{S}\right), 0, 255\right)$$

Operasi Kuantisasi:
$$q = \text{clamp}\left(\text{round}\left(\frac{r}{S} + Z\right), 0, 255\right)$$

Signal-to-Quantization-Noise Ratio:
$$\text{SQNR (dB)} = 10 \log_{10} \left( \frac{\sum r_i^2}{\sum (r_i - \hat{r}_i)^2} \right)$$

In [ ]:
# 2. Setup Lingkungan & Dual-Engine Check
import numpy as np
import sys

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

HAS_TORCH = False
try:
    import torch
    HAS_TORCH = True
    print(f"PyTorch: {torch.__version__} (Device: {'cuda' if torch.cuda.is_available() else 'cpu'})")
except ImportError:
    print("INFO: PyTorch tidak terpasang di host lokal. Beroperasi dalam Mode Analisis Algoritmik NumPy murni.")

In [ ]:
# 3. Implementasi Fungsi Kuantisasi Affine Asimetris
def get_quant_params(r_min, r_max, num_bits=8):
    q_min, q_max = 0, (1 << num_bits) - 1
    safe_min = min(r_min, 0.0)
    safe_max = max(r_max, 0.0)
    scale = (safe_max - safe_min) / float(q_max - q_min)
    raw_z = int(round(-safe_min / scale))
    zero_point = max(q_min, min(q_max, raw_z))
    return scale, zero_point, q_min, q_max

def quantize_affine(vals, scale, zero_point):
    return np.clip(np.round(vals / scale) + zero_point, 0, 255).astype(np.uint8)

def dequantize_affine(q, scale, zero_point):
    return scale * (q.astype(np.float64) - float(zero_point))

# Validasi Parameter Contoh Numerik Seksi 3
s, z, qmin, qmax = get_quant_params(-2.0, 6.0, 8)
print(f"Skala S = {s:.8f} (Expected: {8.0/255.0:.8f})")
print(f"Zero-Point Z = {z} (Expected: 64)")
np.testing.assert_allclose(s, 8.0/255.0, atol=1e-6)
assert z == 64
print("VALIDASI PARAMETER: Berhasil!")

In [ ]:
# 4. Kuantisasi Vektor Sampel & Evaluasi Rekonstruksi SQNR
r_sample = np.array([-2.0, 0.0, 2.0, 6.0], dtype=np.float64)
q_sample = quantize_affine(r_sample, s, z)
r_rec = dequantize_affine(q_sample, s, z)

print(f"Tensor Asli: {r_sample.tolist()}")
print(f"Tensor INT8: {q_sample.tolist()}")
print(f"Tensor Rekonstruksi: {r_rec.tolist()}")

np.testing.assert_array_equal(q_sample, [0, 64, 128, 255])
max_err = np.max(np.abs(r_sample - r_rec))
print(f"Galat Absolut Maksimum: {max_err:.6f} <= {s/2.0:.6f}")
assert max_err <= (s / 2.0) + 1e-5

# Evaluasi SQNR
sqnr = 10.0 * np.log10(np.mean(r_sample**2) / np.mean((r_sample - r_rec)**2))
print(f"SQNR Rekonstruksi: {sqnr:.2f} dB")
assert sqnr > 35.0
print("VALIDASI REKONSTRUKSI & SQNR: Lulus 100%!")

In [ ]:
# 5. Pola Deployment Edge AI: Kompresi Memori & NPU Acceleration
param_count = 1_000_000
fp32_bytes = param_count * 4
int8_bytes = param_count * 1
ratio = fp32_bytes / int8_bytes
savings = (1.0 - int8_bytes / fp32_bytes) * 100.0

print(f"Model Parameter: {param_count:,}")
print(f"FP32 Memory: {fp32_bytes / (1024*1024):.2f} MB")
print(f"INT8 Memory: {int8_bytes / (1024*1024):.2f} MB")
print(f"Faktor Kompresi: {ratio:.1f}x (Penghematan: {savings:.1f}%)")
assert ratio == 4.0
assert savings == 75.0

if HAS_TORCH:
    print("PyTorch Quantization Aware Training (QAT) didukung.")
else:
    print("PANDUAN ALUR KERJA PENYEBARAN EDGE AI:")
    print("1. Simpan checkpoint PyTorch: torch.save(model.state_dict(), 'model.pth')")
    print("2. Konversi ke ONNX / TFLite dengan kalibrasi representatif")
    print("3. Kompilasi ke NPU runtime (misal: edgetpu_compiler model_quant.tflite)")
    print("4. Jalankan inferensi menggunakan akselerasi SIMD / NPU")

### 6. Rangkuman & Pembahasan Akademik
1. **Kompresi 4x Tanpa Kehilangan Makna**: Kuantisasi INT8 memangkas jejak memori sebesar 75% sambil mempertahankan fidelity numerik dengan galat di bawah satu LSB ($< S/2$).
2. **Pentingnya Parameter Zero-Point**: Keberadaan $Z$ menjamin nilai nol (seperti pada ReLU dan zero padding) terpetakan secara eksak tanpa noise buatan.
3. **Kemandirian Sistem Tepi**: Mengeliminasi ketergantungan pada server cloud meningkatkan privasi data visual dan menjamin latensi deterministik rendah untuk sistem kendali otonom.